# ZeroMQ Pipeline Pattern
Auch bekannt als:
- Consumer-Producer
- Push-Pull

Idee:
- Producer (PUSH) verteilt Aufgaben
- mehrere Worker (PULL) holen sich Arbeit
- automatische Lastverteilung

In [1]:
import zmq
import threading
import time
import random

Producer (Push)

In [2]:
def run_producer():
    context = zmq.Context()
    socket = context.socket(zmq.PUSH)

    socket.bind("tcp://127.0.0.1:12345")
    print("🟢 Producer gestartet")

    for i in range(10):
        workload = random.randint(1, 5)
        print(f"📤 Producer sendet Job {i} (Workload: {workload}s)")
        socket.send_string(str(workload))
        time.sleep(0.5)  # simuliert Produktionsrate

    socket.close()
    context.term()
    print("🛑 Producer beendet")

Consumer (Pull)

In [3]:
def run_worker(name):
    context = zmq.Context()
    socket = context.socket(zmq.PULL)

    socket.connect("tcp://localhost:12345")
    print(f"🔵 Worker {name} bereit")

    while True:
        try:
            msg = socket.recv_string()
        except:
            break

        workload = int(msg)
        print(f"⚙️ Worker {name} bearbeitet Job ({workload}s)")

        time.sleep(workload)

        print(f"✅ Worker {name} fertig")

    socket.close()
    context.term()

Mehrere Worker starten

In [4]:
workers = []
for i in range(3):
    t = threading.Thread(target=run_worker, args=(i,), daemon=True)
    workers.append(t)
    t.start()

🔵 Worker 0 bereit
🔵 Worker 1 bereit
🔵 Worker 2 bereit


Producer starten

In [5]:
producer_thread = threading.Thread(target=run_producer, daemon=True)
producer_thread.start()


🟢 Producer gestartet
📤 Producer sendet Job 0 (Workload: 3s)
⚙️ Worker 2 bearbeitet Job (3s)
📤 Producer sendet Job 1 (Workload: 1s)
📤 Producer sendet Job 2 (Workload: 3s)
⚙️ Worker 0 bearbeitet Job (3s)
📤 Producer sendet Job 3 (Workload: 4s)
⚙️ Worker 1 bearbeitet Job (4s)
📤 Producer sendet Job 4 (Workload: 5s)
📤 Producer sendet Job 5 (Workload: 5s)
✅ Worker 2 fertig
⚙️ Worker 2 bearbeitet Job (1s)
📤 Producer sendet Job 6 (Workload: 4s)
📤 Producer sendet Job 7 (Workload: 3s)
✅ Worker 2 fertig
⚙️ Worker 2 bearbeitet Job (5s)
✅ Worker 0 fertig
⚙️ Worker 0 bearbeitet Job (5s)
📤 Producer sendet Job 8 (Workload: 3s)
📤 Producer sendet Job 9 (Workload: 3s)
🛑 Producer beendet
✅ Worker 1 fertig
⚙️ Worker 1 bearbeitet Job (4s)
✅ Worker 2 fertig
⚙️ Worker 2 bearbeitet Job (3s)
✅ Worker 0 fertig
⚙️ Worker 0 bearbeitet Job (3s)
✅ Worker 1 fertig
⚙️ Worker 1 bearbeitet Job (3s)
✅ Worker 2 fertig
✅ Worker 0 fertig
✅ Worker 1 fertig


# Auswertung
Die Jobs werden automatisch verteilt:

Worker 0 → Job
Worker 1 → Job
Worker 2 → Job
Worker 0 → nächster Job
...
-> Round-Robin + Load Balancing Effekt

1. Push verteilt Arbeit
Producer kennt Worker nicht
verteilt automatisch

2. Pull holt Arbeit
Worker fragt aktiv nach Arbeit
kein zentraler Scheduler nötig

3. Load Balancing „for free“
- Schnellere Worker bekommen mehr Jobs
- Langsame werden automatisch entlastet

# Experiment
Worker-Geschwindigkeit ändern:

`time.sleep(workload * 2)  # Worker langsamer machen`  
Beobachtung:
andere Worker übernehmen mehr Jobs
System passt sich automatisch an